# OpenSearch automatic embedding and retrieval

OpenSearch creates document embeddings during ingestion and creates the query
embedding during neural search. Both operations use the same deployed model.

In [1]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

from opensearchpy import OpenSearch, helpers

## 1. Configuration

First run `uv run python -m index_memory_usage.opensearch_ml_model`, then copy
the deployed model ID below.

In [2]:
OPENSEARCH_HOST = "localhost"
OPENSEARCH_PORT = 9200

MODEL_ID = "QEj06qABHp7iMTkBhyaM"
VECTOR_DIMENSION = 384

INDEX_NAME = "brin-datasets-auto-embedding-demo"
INGEST_PIPELINE_NAME = "brin-description-auto-embedding"
SAMPLE_SIZE = 1_000

QUERY_TEXT = "karakteristik morfologi tanaman Piper"
RESULT_COUNT = 5


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Could not find pyproject.toml")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_PATH = PROJECT_ROOT / "scripts" / "scraping" / "data" / "brin" / "datasets.ndjson"

## 2. Load the BRIN dataset

In [3]:
records: list[dict[str, Any]] = []

with DATA_PATH.open(encoding="utf-8") as source:
    for line in source:
        if len(records) >= SAMPLE_SIZE:
            break

        record = json.loads(line)
        if isinstance(record, dict) and record.get("description"):
            records.append(record)

if not records:
    raise RuntimeError(f"No records with descriptions found in {DATA_PATH}")

print(f"Loaded {len(records):,} records from {DATA_PATH.name}.")

Loaded 1,000 records from datasets.ndjson.


## 3. Connect to OpenSearch

In [4]:
client = OpenSearch(
    hosts=[{"host": OPENSEARCH_HOST, "port": OPENSEARCH_PORT}],
    use_ssl=False,
    verify_certs=False,
)

cluster = client.info()
print(f"Connected to OpenSearch {cluster['version']['number']}.")

Connected to OpenSearch 3.8.0.


## 4. Create the automatic embedding pipeline

`field_map` sends `description` to the model and writes its result to
`description_embedding`.

In [5]:
client.transport.perform_request(
    "PUT",
    f"/_ingest/pipeline/{INGEST_PIPELINE_NAME}",
    body={
        "description": "Create BRIN description embeddings during ingestion",
        "processors": [
            {
                "text_embedding": {
                    "model_id": MODEL_ID,
                    "field_map": {
                        "description": "description_embedding",
                    },
                }
            }
        ],
    },
)

{'acknowledged': True}

## 5. Create the vector index

Setting the pipeline as `default_pipeline` makes it run for every document
indexed into this index.

In [6]:
if client.indices.exists(index=INDEX_NAME):
    client.indices.delete(index=INDEX_NAME)

client.indices.create(
    index=INDEX_NAME,
    body={
        "settings": {
            "index": {
                "knn": True,
                "default_pipeline": INGEST_PIPELINE_NAME,
                "number_of_shards": 1,
                "number_of_replicas": 0,
            }
        },
        "mappings": {
            "properties": {
                "name": {"type": "text"},
                "description": {"type": "text"},
                "description_embedding": {
                    "type": "knn_vector",
                    "dimension": VECTOR_DIMENSION,
                    "method": {
                        "name": "hnsw",
                        "space_type": "cosinesimil",
                        "engine": "lucene",
                    },
                },
                "url": {"type": "keyword", "index": False},
                "global_id": {"type": "keyword"},
            }
        },
    },
)

{'acknowledged': True,
 'shards_acknowledged': True,
 'index': 'brin-datasets-auto-embedding-demo'}

## 6. Index documents without creating embeddings in Python

None of these source documents contains `description_embedding`. The default
ingest pipeline adds that field before OpenSearch indexes each document.

In [8]:
actions = [
    {
        "_op_type": "index",
        "_index": INDEX_NAME,
        "_id": record.get("global_id") or str(position),
        "_source": {
            "name": record.get("name"),
            "description": record["description"],
            "url": record.get("url"),
            "global_id": record.get("global_id"),
        },
    }
    for position, record in enumerate(records, start=1)
]

indexed_count, _ = helpers.bulk(client, actions, refresh=True)
print(f"Indexed {indexed_count:,} documents into {INDEX_NAME!r}.")

Indexed 1,000 documents into 'brin-datasets-auto-embedding-demo'.


Confirm that OpenSearch added the embedding during ingestion.

In [9]:
sample_document = client.search(
    index=INDEX_NAME,
    body={"size": 1, "query": {"match_all": {}}},
)["hits"]["hits"][0]["_source"]

print(
    "Generated embedding dimensions: "
    f"{len(sample_document['description_embedding']):,}"
)

Generated embedding dimensions: 384


## 7. Query with automatic query embedding

The `neural` query sends `query_text` to the same model and searches the
resulting vector against `description_embedding`.

In [10]:
def search(query_text: str) -> list[dict[str, Any]]:
    response = client.search(
        index=INDEX_NAME,
        body={
            "size": RESULT_COUNT,
            "_source": {"excludes": ["description_embedding"]},
            "query": {
                "neural": {
                    "description_embedding": {
                        "query_text": query_text,
                        "model_id": MODEL_ID,
                        "k": RESULT_COUNT,
                    }
                }
            },
        },
    )
    return response["hits"]["hits"]


print(f"\nQuery: {QUERY_TEXT}")
for rank, hit in enumerate(search(QUERY_TEXT), start=1):
    source = hit["_source"]
    print(f"{rank}. score={hit['_score']:.4f} | {source['name']}")
    print(f"   {source['description'][:200]}...")


Query: karakteristik morfologi tanaman Piper
1. score=0.9233 | Karakteristik Morfologi Piper Acutilimbum C. DC. dan Piper Baccatum Blume
   Piper acutilimbum C. DC dan Piper baccatum Blume merupakan dua jenis Piper yang bentuk morfologinya hampir sama, terutama pada fase mudanya. Kedua jenis Piper ini merupakan tanaman koleksi Kebun Raya ...
2. score=0.8988 | Informasi Species Baru  Piper Sp Dan Studi Perbanyakan Pada Tumbuhan Hias Sirih (Piper Crocatum Dan Piper Sp.).
   Red piper (Piper crocatum) and unidentified ornamental piper (Piper sp.) are cultivated by developed technique. Identification of key to species of Piper sp. described clearly that both species close ...
3. score=0.8518 | Penggalian Potensi Piper spp. dari Suaka Alam Maninjau Utara Selatan dan Batang Pagean II Sebagai Tanaman Hias dan Obat
   Potential Piper spp as an ornamental and medicinal plants was conducted based on morphological characteristic and atsiri oil aromatic of this leaves. The morphological character